In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import joblib
import shap 

plt.rcParams['figure.figsize'] = (9, 6)
plt.rcParams['font.size'] = 12
sns.set_theme(style="whitegrid")

/home/kota/my_central_venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# загрузка артефактов
model = joblib.load('models/best_model.pkl')
scaler = joblib.load('models/scaler.pkl')
le = joblib.load('models/label_encoder.pkl')

# загружаем данные 
df = pd.read_csv('data/processed/spotify_clean.csv')
X = df.drop('track_genre', axis=1)
y_raw = df['track_genre']
y = le.transform(y_raw)
class_names = le.classes_

# делаем масштабирование
X_scaled = scaler.transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print(f"Загружено: {len(X_scaled_df)} треков, {len(class_names)} жанров")
print(f"Модель: {model.__class__.__name__}")

Загружено: 9987 треков, 10 жанров
Модель: RandomForestClassifier


In [ ]:
explainer = shap.TreeExplainer(model) # создаем explainer спецаильно для деревьев

shap_values = explainer.shap_values(X_scaled_df) # считаем для каждого трека и признака, насколько они влияют на предсказание

print('Строим глобальный график важности:') # усреднен по всем трекам и классам 
shap.summary_plot(shap_values, X_scaled_df, plot_type='bar', show = False)
plt.title('Глобальная важность признаков')
plt.xlabel('Средняя абсолютная велчина SHAP')
plt.tight_layout()
plt.show()

In [ ]:
# локальное объяснение
# выбираем конкретный трек
idx = 10
row = X_scaled_df.iloc[idx:idx+1]
pred_clases = model.predict(row)[0]
pred_genre = le.inverse_transform([pred_clases]) 

print(f"Трек #{idx} | Предсказание модели: {pred_genre[0]}")

shap_vals_for_class = shap_values[pred_clases][idx] # извлекаем SHAP value для нашего класса
# строим график того, как модель пришла к предсказанию
shap.plots.waterfall(
    shap.Explanation(
        values = shap_vals_for_class,
        base_values = explainer.expected_value[pred_clases],  
        data = row.iloc[0],
        feature_names = row.columns
    ),
    max_display = 10
)